# Puffin Transcription Initiation

![Puffin Transcription Initiation](https://proto-bio.github.io/proto-assets/images/tool/puffin/hero.png)

Puffin predicts per-base transcription initiation signals from DNA sequence and decomposes them into learned motif contributions ([Dudnyk et al., *Science* 2024](https://doi.org/10.1126/science.adj0116)). Both tools share the same model: `puffin-prediction` is a fast forward pass returning all 10 channels, and `puffin-interpretation` runs gradient-based motif decomposition for one chosen target signal and strand.

In [1]:
from proto_tools.utils.notebook_docs import display_api_reference, display_available_tools, display_doc_link, display_docs_section, display_overview

display_doc_link("puffin")
display_overview("puffin")
display_docs_section("puffin", "Background")

# Puffin

Puffin is a sequence-based deep learning model for transcription initiation that predicts per-base initiation signal from DNA and decomposes the prediction into a small set of learned promoter motifs. This toolkit exposes a fast prediction path (`puffin-prediction`) and a gradient-based motif-decomposition path (`puffin-interpretation`) over the same checkpoint.

In 2024, [Dudnyk et al.](https://doi.org/10.1126/science.adj0116) introduced Puffin, a deep learning model that explains transcription initiation in the human genome at basepair resolution from sequence alone. The model is trained against five [transcription initiation](https://en.wikipedia.org/wiki/Transcription_(biology)) assays (FANTOM CAGE, ENCODE CAGE, ENCODE RAMPAGE, GRO-cap, PRO-cap), each predicted on both strands. The output is a per-base 10-channel signal that can be interpreted as `ln(count_scale_signal + 1)`.

Puffin is structurally constrained: its first convolutional layer plays the role of a learned [motif](https://en.wikipedia.org/wiki/Sequence_motif) filter bank, and the model exposes per-base activation and contribution scores for nine promoter motifs (CREB, ETS, NFY, NRF1, SP, TATA, U1_snRNP, YY1, ZNF143) on each strand. A tenth `Long Inr` filter is used internally by the model to construct the per-base initiator-effect track but is not exposed per-motif. The minimum input is 651 bp because the model uses 325 bp of padding on each side of the predicted output span.

The wrapper accepts raw DNA strings; the upstream coordinate / region / FASTA-file CLI modes (which require an hg38 reference) are intentionally not exposed and callers extract genomic sequences themselves.

## Available tools

In [2]:
display_available_tools("puffin")

- **`run_puffin_interpretation()`** — Motif-level interpretation of transcription initiation with Puffin
- **`run_puffin_prediction()`** — Basepair-resolution transcription initiation prediction with Puffin

### `run_puffin_prediction`

Fast forward pass through Puffin. For each input DNA sequence (>= 651 bp) returns per-base predictions across 10 channels (five transcription assays on both strands). Per-base output length equals `len(sequence) - 650`.

In [3]:
from proto_tools import (
    PuffinPredictionConfig,
    PuffinPredictionInput,
    TRACK_NAMES,
    run_puffin_prediction,
)

In [4]:
# Display input docs
display_api_reference("puffin-prediction", "input", "run_puffin_prediction")

**Input** — `PuffinPredictionInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | DNA sequence(s) >= 651 bp. Per-base output length = len(seq) - 650 |

In [5]:
# Display config docs
display_api_reference("puffin-prediction", "config", "run_puffin_prediction")

**Config** — `PuffinPredictionConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |

In [6]:
from pathlib import Path

# Real HBB promoter window (1650 bp -> 1000 bp per-base output); the HBB
# transcription start site sits at the centre of the scored region.
hbb = "".join(l for l in Path("hbb_genomic.fasta").read_text().splitlines() if not l.startswith(">"))
sequence = hbb[4176:5826]
prediction_result = run_puffin_prediction(
    PuffinPredictionInput(sequences=[sequence]),
    PuffinPredictionConfig(),
)

entry = prediction_result.results[0]
fantom_idx = TRACK_NAMES.index("FANTOM_CAGE+")
peak = max(range(entry.output_length), key=lambda i: entry.predictions[i][fantom_idx])
print(f"shape: {entry.output_length} bp x {len(entry.predictions[0])} channels")
print(f"FANTOM_CAGE+ peak at input position {entry.output_start + peak}")

Running run_puffin_prediction [00:00]

shape: 1000 bp x 10 channels
FANTOM_CAGE+ peak at input position 824


In [7]:
display_api_reference("puffin-prediction", "output", "run_puffin_prediction")

**Output** — `PuffinPredictionOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[PuffinPredictionResult]</code> | required | Per-sequence Puffin prediction results |
| <code>track_names</code> | <code>list[str]</code> | required | Names of the 10 output channels in order |

**`PuffinPredictionResult`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequence</code> | <code>str</code> | required | DNA sequence originally provided to the tool |
| <code>sequence_length</code> | <code>int</code> | required | Length of the provided DNA sequence |
| <code>output_length</code> | <code>int</code> | required | Number of per-base output positions (= sequence_length - 650) |
| <code>output_start</code> | <code>int</code> | required | 0-based sequence coordinate of the first output position |
| <code>output_end</code> | <code>int</code> | required | 0-based exclusive end of the output span in the input sequence |
| <code>predictions</code> | <code>list[list[float]]</code> | required | Per-base log-scale predictions in TRACK_NAMES order (shape [output_length, 10]) |

### `run_puffin_interpretation`

Gradient-based motif decomposition for one chosen target signal and strand. Returns the per-base prediction plus 18 motif activation tracks (9 motifs x 2 strands) and basepair-level contribution scores.

In [8]:
from proto_tools import (
    MOTIF_NAMES,
    PuffinInterpretationConfig,
    PuffinInterpretationInput,
    run_puffin_interpretation,
)

In [9]:
# Display input docs
display_api_reference("puffin-interpretation", "input", "run_puffin_interpretation")

**Input** — `PuffinInterpretationInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | DNA sequence(s) >= 651 bp. Per-base output length = len(seq) - 650 |

In [10]:
# Display config docs
display_api_reference("puffin-interpretation", "config", "run_puffin_interpretation")

**Config** — `PuffinInterpretationConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>target_signal</code> | <code>Literal['FANTOM_CAGE', 'ENCODE_CAGE', 'ENCODE_RAMPAGE', 'GRO_CAP', 'PRO_CAP']</code> | <code>'FANTOM_CAGE'</code> | Which transcription initiation assay's predictions to decompose |
| <code>reverse_strand</code> | <code>bool</code> | <code>False</code> | Decompose the reverse-strand prediction instead of the forward strand |

In [11]:
# Decompose the PRO_CAP forward-strand prediction
sequence = "ATCG" * 412 + "AT"
interpretation_result = run_puffin_interpretation(
    PuffinInterpretationInput(sequences=[sequence]),
    PuffinInterpretationConfig(target_signal="PRO_CAP"),
)

r = interpretation_result.results[0]
print(f"target_signal: {interpretation_result.target_signal}, reverse_strand: {interpretation_result.reverse_strand}")
print(f"motif tracks: {len(r.motif_activations)} (9 motifs x 2 strands)")
print(f"motifs: {MOTIF_NAMES}")

Running run_puffin_interpretation [00:00]

target_signal: PRO_CAP, reverse_strand: False
motif tracks: 18 (9 motifs x 2 strands)
motifs: ('CREB', 'ETS', 'NFY', 'NRF1', 'SP', 'TATA', 'U1_snRNP', 'YY1', 'ZNF143')


In [12]:
display_api_reference("puffin-interpretation", "output", "run_puffin_interpretation")

**Output** — `PuffinInterpretationOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[PuffinInterpretationResult]</code> | required | Per-sequence interpretation results |
| <code>target_signal</code> | <code>str</code> | required | Target transcription initiation signal that was interpreted |
| <code>reverse_strand</code> | <code>bool</code> | required | Whether the reverse-strand head was used |
| <code>motif_names</code> | <code>list[str]</code> | required | The 9 learned motif names (without strand suffix) |

**`PuffinInterpretationResult`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequence</code> | <code>str</code> | required | DNA sequence originally provided to the tool |
| <code>sequence_length</code> | <code>int</code> | required | Length of the provided DNA sequence |
| <code>output_length</code> | <code>int</code> | required | Number of per-base output positions (= sequence_length - 650) |
| <code>output_start</code> | <code>int</code> | required | 0-based sequence coordinate of the first output position |
| <code>output_end</code> | <code>int</code> | required | 0-based exclusive end of the output span in the input sequence |
| <code>prediction</code> | <code>list[float]</code> | required | Predicted log-scale signal for the selected target and strand, length output_length |
| <code>motif_activations</code> | <code>dict[str, list[float]]</code> | required | Per-base motif activation scores keyed by strand-suffixed motif name |
| <code>motif_effects</code> | <code>dict[str, list[float]]</code> | required | Per-base motif effect scores keyed by strand-suffixed motif name |
| <code>sum_motif_effects</code> | <code>list[float]</code> | required | Per-base sum of non-initiator motif effects |
| <code>sum_initiator_effects</code> | <code>list[float]</code> | required | Per-base sum of initiator-motif effects, centered |
| <code>sum_trinucleotide_effects</code> | <code>list[float]</code> | required | Per-base sum of trinucleotide sequence effects, centered |
| <code>sum_total_effects</code> | <code>list[float]</code> | required | Per-base sum of motif, initiator, and trinucleotide effects |
| <code>bp_contribution</code> | <code>list[float]</code> | required | Per-base contribution to transcription initiation |
| <code>bp_contribution_per_motif</code> | <code>dict[str, list[float]]</code> | required | Per-base contribution to transcription, decomposed by motif |
| <code>bp_contribution_to_motif_activation</code> | <code>dict[str, list[float]]</code> | required | Per-base contribution to motif-activation scores, decomposed by motif |

## Export Results

In [13]:
from pathlib import Path

out_dir = Path("puffin_output")
out_dir.mkdir(exist_ok=True)
prediction_result.export(name="prediction", export_path=out_dir, file_format="json")
interpretation_result.export(name="interpretation", export_path=out_dir, file_format="json")